In [1]:
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.optim as optim
import re
import math

# Data Cleaning And Tokenization

In [2]:
text = """
i love artificial intelligence and machine learning.
artificial intelligence is transforming the world.
machine learning allows computers to learn from data.
deep learning is a part of machine learning.
ai is powerful and exciting.
"""

# text = text.split() better way but now ew are working i

In [3]:
chars = sorted(list(set(text)))
print(chars)

['a', 'ai', 'allows', 'and', 'artificial', 'computers', 'data.', 'deep', 'exciting.', 'from', 'i', 'intelligence', 'is', 'learn', 'learning', 'learning.', 'love', 'machine', 'of', 'part', 'powerful', 'the', 'to', 'transforming', 'world.']


In [4]:
print(f"Total Unque Vocab {len(chars)}")

Total Unque Vocab 25


In [5]:
word_to_int = {}

int_to_word = {}

for i,data in enumerate(chars):
    word_to_int[data] = i


for i,data in enumerate(chars):
    int_to_word[i] = data

    

In [6]:
word_to_int

{'a': 0,
 'ai': 1,
 'allows': 2,
 'and': 3,
 'artificial': 4,
 'computers': 5,
 'data.': 6,
 'deep': 7,
 'exciting.': 8,
 'from': 9,
 'i': 10,
 'intelligence': 11,
 'is': 12,
 'learn': 13,
 'learning': 14,
 'learning.': 15,
 'love': 16,
 'machine': 17,
 'of': 18,
 'part': 19,
 'powerful': 20,
 'the': 21,
 'to': 22,
 'transforming': 23,
 'world.': 24}

In [7]:
int_to_word

{0: 'a',
 1: 'ai',
 2: 'allows',
 3: 'and',
 4: 'artificial',
 5: 'computers',
 6: 'data.',
 7: 'deep',
 8: 'exciting.',
 9: 'from',
 10: 'i',
 11: 'intelligence',
 12: 'is',
 13: 'learn',
 14: 'learning',
 15: 'learning.',
 16: 'love',
 17: 'machine',
 18: 'of',
 19: 'part',
 20: 'powerful',
 21: 'the',
 22: 'to',
 23: 'transforming',
 24: 'world.'}

# Encoding and decoding (Testing)

In [8]:
def encoder(data):
    result = []
    for char in data:
         result.append(word_to_int[char])
    return result
    

def decoder(data):
    result = " "
    for char in data:
        result+=int_to_word[char]

    return result
    

In [9]:
# Testing

encoded_value = encoder("deep learning is a part of machine learning.")
print(encoded_value)

KeyError: 'd'

In [ ]:
# Decoding 
decoded_value = decoder(encoded_value)
print(decoded_value)

# Embadding values

In [ ]:
def embadding(input_tensor, vocab_size, d_model=5):
    embedding = nn.Embedding(vocab_size, d_model)
    return embedding(input_tensor)

In [ ]:
vocab_size = len(chars)
encoded = encoder(text)
input_tensor = torch.tensor(encoded).unsqueeze(0)

embedded = embadding(input_tensor, vocab_size)

print(embedded.shape)
print(embedded)

# Postional Encodeere

In [ ]:
# When we convert words into embeddings (numbers), we lose the order of words.

# Example:

# "lion eats human"
# "human eats lion"
# "eat lion human"

# All contain same words, but meaning is different

 # Problem

# Embeddings only capture:

# meaning of words  but NOT their position 

 # So model cannot understand:

# who is doing action
# who is receiving action
#  Solution: Positional Encoding

# To solve this, we add extra information about position.

#  This is done using:

# sine (sin)
# cosine (cos) functions

# with different frequencies.


# Affect after adding Positional encoder 
#  no of dimension  of postional  == no of embadding dimension  
#  Small changes in position lead to smooth and consistent changes in the encoding vectors, which helps the model learn relationships between nearby words effectively.

In [ ]:
# Workflow

# We have a sentence: "hello world"

# Step 1: Embedding
# hello → [0.0, 0.1, 0.9]
# world → [0.1, 0.5, 1.0]

# Step 2: Positional Encoding
# We DO NOT apply sin and cos to embedding values.

# Instead, we generate positional vectors using sin and cos based on position index:

# pos 0 → [sin(0), cos(0), ...]
# pos 1 → [sin(1), cos(1), ...]

# Step 3: Add them
# hello_final = embedding(hello) + PE(pos=0)
# world_final = embedding(world) + PE(pos=1)

# So, number of embeddings = number of positions (sequence length)

# Each word gets:
# 1 embedding vector + 1 positional encoding vector

# Embedding = what the word is
# Position = where the word is
# Final = what + where

In [ ]:
def positional_embad(seq_len,embadded_dim):  #seq len give total sentence contain like hello world  = seq_len = 2
    pe = torch.zeros(seq_len,embadded_dim)

    for pos in range(seq_len):

        for i in range(0,embadded_dim,2):  #genreate even number
            angle = pos / (10000 ** (i / embadded_dim))
            # Now assigning Sin in even and cos in odd
            
            pe[pos,i] = math.sin(angle)

            # so evert time odd vlaue or index may not exist
            if i+1 <embadded_dim:
                pe[pos,i+1] = math.cos(angle)


    return pe
                
            

In [ ]:
# debug + Test
embedded.shape

In [ ]:
# Now cancatting embadded + Position

seq_len = embedded.shape[1]  #total length in 1 sentence
embedded_dim =embedded.shape[2]  


position_emb = positional_embad(seq_len,embedded_dim)

position_emb




In [ ]:
final_embadded = embedded + position_emb

In [ ]:
print(f"Dim is Embadded value {embedded.shape}")
print()
print(f"Positional Embadded value {position_emb.shape}")
print()
print(f"Embadded + Positional value : {final_embadded.shape}")

In [ ]:
final_embadded


# Now Self Attention

In [ ]:
def self_attention(embadded_dim,final_embadded):

    wq = nn.Linear(embadded_dim,embadded_dim)
    wk = nn.Linear(embadded_dim,embadded_dim)
    wv = nn.Linear(embadded_dim,embadded_dim)
    
    
    q_matrix = wq(final_embadded)
    k_matrix = wk(final_embadded)
    v_matrix= wv(final_embadded)


    # Now qkT
    kT = k_matrix.transpose(-2,-1)

    # Scaling with Dot Product to reduct large value cause large value and when large value feed to softmax we get value close to 1 or 0 enf to end value

    scores = q_matrix@kT

    # now scaling
    scores = scores/math.sqrt(embadded_dim)


    attention_wts = torch.softmax(scores,dim=-1)  #this provide probability range which say how much to focus on each word
    
    # now 
    output = attention_wts@v_matrix

    return output
    

In [ ]:
self_at = self_attention(embedded_dim,final_embadded)
self_at

In [ ]:
self_at.shape

# Multi- Head Attention

In [ ]:
# Multi head atteintion is same as Self attention but  different is for single attention we called single head attaention
# for multiple attention we call it a Multi head Attention


In [ ]:
# Debug
print(embedded_dim)
print(final_embadded.shape)

In [ ]:
# head_dim	features per head
# num_heads	number of splits
def multi_head_attention(embadded_dim,final_embadded,num_heads):
    head_dim = embadded_dim // num_heads
    # wq = torch.arange(embadded_dim,embadded_dim)
    if embadded_dim % num_heads == 0:
        wq = nn.Linear(embadded_dim,embadded_dim)  #cause embadded and and final have same dim so can use any
        wk = nn.Linear(embadded_dim,embadded_dim)
        wv = nn.Linear(embadded_dim,embadded_dim)

        q = wq(final_embadded)
        k = wk(final_embadded)
        v = wv(final_embadded)


        batch_size,seq_len,embadded_dim = final_embadded.shape

        # splitting embadded_dim into num_head and head_dim
        q = q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)  # seq tokens each token now has num_head heads each head has head_dim features
        k = k.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)  #(batch, seq_len, num_heads, head_dim) -------> (batch, num_heads, seq_len, head_dim)
        v = v.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)


        kT = k.transpose(-2,-1)

        # scores = q @ kT
        scores = q @ kT
        scores = scores / math.sqrt(head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous() #(batch, num_heads, seq_len, head_dim) -----> (batch, seq_len, num_heads, head_dim)  #contiguous mean data is stored in continious memory
        out = out.view(batch_size, seq_len, embadded_dim)

        return out

    else:
        print("Num head and nead dim embadded_dim % num_heads == 0:")

        

In [ ]:
multihead_at = multi_head_attention(embedded_dim,final_embadded,1)
multihead_at

# Add and Normalization

In [ ]:
def add_norm(final_emb, attention_output, embadded_dim):

    norm = nn.LayerNorm(embadded_dim)

    x = final_emb + attention_output # residual connection  x skip connection  att_op op from attention
    x = norm(x)               # normalization

    return x

In [ ]:
x = add_norm(final_embadded,multihead_at, embedded_dim)
x


# Feed Forward NN

In [ ]:
def ff_neuralnetwork(x,embadded_dim):
    model = nn.Sequential(nn.Linear(embadded_dim,65),
                          nn.ReLU(),
                          nn.Linear(65,embadded_dim))
    return model(x)
    

In [ ]:
ff_nn = ff_neuralnetwork(x,embedded_dim)
ff_nn